## Previsão de Tendência do IBOVESPA
### Tech Challenge - Fase 02

Este notebook desenvolve um modelo preditivo para prever se o índice IBOVESPA fechará em **alta (↑) ou baixa (↓)** no dia seguinte, com base em dados históricos do próprio índice.

O problema é tratado como uma tarefa de **classificação binária**, onde:
- `1` → fechamento em alta em relação ao dia anterior
- `0` → fechamento em baixa em relação ao dia anterior

Os dados utilizados cobrem o período de **março/2024 a março/2026** (500 pregões), obtidos do portal Investing.com.

In [897]:
import pandas as pd
import numpy as np

### 1. Aquisição e Exploração dos Dados

Os dados históricos do IBOVESPA foram obtidos publicamente em [br.investing.com](https://br.investing.com/indices/bovespa-historical-data), selecionando o período diário com intervalo mínimo de 2 anos.

As colunas disponíveis são:
- **Data**: data do pregão
- **Último**: preço de fechamento
- **Abertura**: preço de abertura
- **Máxima / Mínima**: extremos do dia
- **Vol.**: volume negociado
- **Var%**: variação percentual do dia

O índice de data foi convertido para `DatetimeIndex` e os dados foram ordenados cronologicamente para garantir a integridade da série temporal.

In [898]:
from pathlib import Path

#caminho = Path("../data/raw/Dados Históricos_Wesley.csv")
caminho = Path("../data/raw/IBOVESPA_historicos_032016_032026.csv")
df = pd.read_csv(caminho, parse_dates=[0], index_col=0, decimal=",", thousands=".") # Alteração: ajustando a colunda de data e colocando como index 
df.index = pd.to_datetime(df.index, format = "%d.%m.%Y")
df = df.sort_index()

/var/folders/f3/f42x8rcn6ml34_jb4_l343r80000gn/T/ipykernel_66131/397529687.py:5: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv(caminho, parse_dates=[0], index_col=0, decimal=",", thousands=".") # Alteração: ajustando a colunda de data e colocando como index


In [899]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2499 entries, 2016-03-01 to 2026-03-20
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Último    2499 non-null   int64
 1   Abertura  2499 non-null   int64
 2   Máxima    2499 non-null   int64
 3   Mínima    2499 non-null   int64
 4   Vol.      2499 non-null   str  
 5   Var%      2499 non-null   str  
dtypes: int64(4), str(2)
memory usage: 136.7 KB


### 2. Definição do Target

O objetivo é prever a **tendência do dia seguinte**, não o valor exato do índice.

Para tornar o problema mais robusto, foi adotado um **threshold de 0,3%**: apenas dias com variação superior a esse valor em magnitude são considerados como alta ou baixa clara. Dias com variação muito pequena são descartados, pois representam ruído de mercado sem sinal preditivo relevante.

| Condição | Target |
|---|---|
| Variação > +0,3% | 1 → Alta |
| Variação < -0,3% | 0 → Baixa |
| -0,3% ≤ Variação ≤ +0,3% | Descartado (ruído) |

Essa decisão aumentou a separabilidade entre as classes e melhorou a performance dos modelos.

In [900]:
# retorno futuro (amanhã vs hoje)
retorno = df["Último"].shift(-1) / df["Último"] - 1

# Threshold: só considera alta/baixa se a variação for maior que 0.3%
# Foi criada uma zona neutra de variação inferior a 0.3% para reduzir ruído direcional típico de séries financeiras diárias.
threshold = 0.003

df["y"] = np.where(retorno > threshold, 1,
          np.where(retorno < -threshold, 0, np.nan))

# remove o que for do target
df = df.dropna(subset=["y"])

print("Distribuição do novo target:")
print(df["y"].value_counts(normalize=True).round(3))
print("Dias descartados (ruído):", retorno.notna().sum() - df["y"].notna().sum())

Distribuição do novo target:
y
1.0    0.542
0.0    0.458
Name: proportion, dtype: float64
Dias descartados (ruído): 537


### 3. Engenharia de Atributos (Feature Engineering)

Os dados brutos de preço foram transformados em variáveis que capturam **padrões de comportamento do mercado**: tendência, momentum, volatilidade e pressão compradora/vendedora.

**Princípio fundamental:** todas as features são defasadas em um período (`shift(1)`), garantindo que o modelo utiliza apenas informações disponíveis **até o fechamento do dia anterior**. Isso evita data leakage — situação em que o modelo "vê o futuro" durante o treinamento.

### Retornos Recentes

In [901]:
df["retorno_1"] = df["Último"].pct_change().shift(1)
df["retorno_3"] = df["Último"].pct_change(3).shift(1)

- **retorno_1**: variação percentual do pregão imediatamente anterior
- **retorno_3**: variação acumulada dos últimos 3 pregões — captura momentum de curtíssimo prazo
- **vol_5 / vol_10**: desvio padrão dos retornos nas últimas 5 e 10 sessões — mede o nível de volatilidade recente do mercado

In [902]:
df["vol_5"] = df["Último"].pct_change().rolling(5).std().shift(1)
df["vol_10"] = df["Último"].pct_change().rolling(10).std().shift(1)

### Médias Móveis

In [903]:
# Tendência de curto e médio prazo baseada em médias móveis defasadas.
df["ma_5"] = df["Último"].rolling(5).mean().shift(1)
df["ma_10"] = df["Último"].rolling(10).mean().shift(1)

# Diferença relativa entre a média móvel de curto prazo (5 dias) e longo prazo (10 dias).
# Utilizar a forma percentual (e não a diferença em pontos) permite capturar a intensidade
# da tendência de forma mais robusta, evitando dependência do nível absoluto do IBOVESPA
# e melhorando a generalização do modelo ao longo do tempo.
df["ma_diff_rel"] = (df["ma_5"] / df["ma_10"] - 1)

- **MA5** → média dos últimos 5 pregões, usada como proxy de tendência de curtíssimo prazo  
- **MA10** → média dos últimos 10 pregões, usada como proxy de tendência de curto prazo  
- **ma_diff_rel** → diferença relativa entre MA5 e MA10, capturando a intensidade da tendência em termos percentuais

### Distâncias Relativas, Range e Pressão de Fechamento

In [904]:
# calculando a diferença de range
df["range_diff"] = (df["Máxima"] - df["Mínima"]).shift(1)

In [905]:
# 1. Distâncias Relativas (Substituindo ma_5 e ma_10 brutos)
df["dist_ma_5"] = (df["Último"] / df["ma_5"] - 1)
df["dist_ma_10"] = (df["Último"] / df["ma_10"] - 1)

# 2. Pressão de Compra/Venda (Onde fechou dentro do candle anterior)
df["pressao_fechamento"] = ((df["Último"] - df["Mínima"]) / (df["Máxima"] - df["Mínima"] + 1e-9)).shift(1)

- **range_diff**: amplitude do candle anterior (máxima − mínima). Candles largos indicam sessões de alta volatilidade e indecisão.
- **dist_ma_5 / dist_ma_10**: distância percentual do preço em relação a cada média móvel. Positivo = preço acima da média (tendência de alta); negativo = abaixo.
- **pressao_fechamento**: posição do fechamento dentro do range do candle anterior. Próximo de 1 = fechou perto da máxima (pressão compradora); próximo de 0 = perto da mínima (pressão vendedora).

### Momentum

Diferença entre o preço atual e o preço de 5 pregões atrás. Captura a força e direção do movimento recente — valores positivos indicam tendência de alta sustentada; negativos, queda acumulada.

In [906]:
df["momentum_5"] = (df["Último"] - df["Último"].shift(5)).shift(1)

In [907]:
# As primeiras linhas da série é removida devido à ausência de um valor anterior para cálculo da diferença.
# removendo linhas sem target ou sem features do modelo
df = df.dropna(subset=[
    "y", "retorno_1", "retorno_3", "momentum_5",
    "range_diff", "dist_ma_5", "dist_ma_10",
    "ma_diff_rel", "pressao_fechamento"
])

### 4. Separação Treino e Teste

Em séries temporais, a divisão dos dados **nunca pode ser aleatória**. Usar amostras futuras no treino causaria vazamento de informação e resultados artificialmente inflados.

A estratégia adotada foi a **divisão temporal**: os últimos 30 pregões (aproximadamente 1 mês de mercado) foram reservados para teste, e todos os dados anteriores foram usados para treino. Isso replica fielmente o cenário real de uso do modelo.

| Conjunto | Período | Tamanho |
|---|---|---|
| Treino | Mar/2024 — Dez/2025 | ~430 dias |
| Teste | Jan/2026 — Mar/2026 | 30 dias |

In [908]:
# criando base de teste com as ultimas 30 linhas
test = df.tail(30)
test

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10,ma_5,ma_10,ma_diff_rel,range_diff,dist_ma_5,dist_ma_10,pressao_fechamento,momentum_5
Data,,,,,,,,,,,,,,,,,,,
2026-01-22,175589,171817,177742,171817,"14,02B","2,20%",1.0,0.033318,0.037743,0.017284,0.013688,166096.8,164168.2,0.011748,5691.0,0.057149,0.069568,0.973291,8667.0
2026-01-26,178721,178859,179543,177694,"9,57B","-0,08%",1.0,0.021954,0.065151,0.014498,0.013966,168820.0,165673.2,0.018994,5925.0,0.058648,0.078756,0.636624,13616.0
2026-01-27,181919,178852,183360,178852,"11,34B","1,79%",1.0,0.017837,0.074839,0.014187,0.014229,171450.6,167358.3,0.024452,1849.0,0.061058,0.087003,0.555435,13153.0
2026-01-28,184691,181921,185065,181921,"11,86B","1,52%",0.0,0.017894,0.058795,0.008928,0.014446,174864.6,169183.8,0.033578,4508.0,0.056194,0.091659,0.680346,17070.0
2026-01-29,183134,184692,186450,181567,"12,95B","-0,84%",0.0,0.015238,0.051837,0.007162,0.012429,178547.4,171455.4,0.041364,3144.0,0.025688,0.068115,0.881043,18414.0
2026-01-30,181364,183133,183620,180089,"11,11B","-0,97%",1.0,-0.008430,0.024692,0.012163,0.014145,180810.8,173453.8,0.042415,4883.0,0.003060,0.045604,0.320909,11317.0
2026-02-02,182793,181369,182890,181348,"8,74B","0,79%",1.0,-0.009665,-0.003051,0.014308,0.014525,181965.8,175392.9,0.037475,3531.0,0.004546,0.042192,0.361088,5775.0
2026-02-03,185674,182816,187334,182816,"10,76B","1,58%",0.0,0.007879,-0.010277,0.012980,0.014048,182780.2,177115.4,0.031984,1542.0,0.015832,0.048322,0.937095,4072.0
2026-02-05,182127,181708,184017,181569,"9,83B","0,23%",1.0,0.015761,0.013870,0.012458,0.013173,183531.2,179197.9,0.024182,4518.0,-0.007651,0.016346,0.632581,3755.0


In [909]:
# montando base de treino sem as ultimas 30 linhas
train = df.iloc[:-30]
train

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10,ma_5,ma_10,ma_diff_rel,range_diff,dist_ma_5,dist_ma_10,pressao_fechamento,momentum_5
Data,,,,,,,,,,,,,,,,,,,
2016-03-18,50815,50915,51308,50202,"5,62M","-0,19%",1.0,0.013431,-0.037793,0.022497,NaN,48412.8,47645.9,0.016096,1293.0,0.049619,0.066514,0.960557,-1339.0
2016-03-21,51172,50816,51370,50765,"3,82M","0,70%",0.0,0.063899,0.039863,0.037887,0.030984,48842.8,48315.2,0.010920,1106.0,0.047688,0.059128,0.554250,2150.0
2016-03-22,51010,51170,51215,50812,"4,02M","-0,32%",0.0,0.007025,0.085763,0.037406,0.031054,49149.4,48943.1,0.004215,605.0,0.037856,0.042231,0.672727,1533.0
2016-03-24,49657,49686,49686,48778,"3,80M","-0,07%",1.0,-0.003166,0.067981,0.035950,0.028380,49578.0,49324.8,0.005133,403.0,0.001593,0.006735,0.491315,2143.0
2016-03-28,50838,49687,51149,49687,"3,62M","2,38%",1.0,-0.026524,-0.022789,0.033274,0.027867,50083.4,49382.0,0.014204,908.0,0.015067,0.029484,0.968062,2527.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-13,161973,163146,163146,161765,"9,67B","-0,72%",1.0,0.007254,0.007908,0.009108,0.008136,162239.6,160620.5,0.010080,1216.0,-0.001643,0.008420,0.717928,2025.0
2026-01-15,165568,165180,166070,164833,"9,41B","0,26%",0.0,-0.007214,-0.010332,0.009826,0.008039,162526.4,161085.1,0.008947,1381.0,0.018714,0.027829,0.150615,1434.0
2026-01-19,164849,164800,165155,164265,"4,80B","0,03%",1.0,0.022195,0.022182,0.013424,0.010098,163266.0,161849.6,0.008751,1237.0,0.009696,0.018532,0.594179,3698.0


In [910]:
# Definição das variáveis de entrada
features = [
    "retorno_1",
    "retorno_3",
    "momentum_5",
    "range_diff",
    "dist_ma_5",
    "dist_ma_10",
    "ma_diff_rel",
    "pressao_fechamento"
]

# Definindo X e y de treino
X_train = train[features]
y_train = train["y"]

# Definindo X e y de teste
X_test = test[features]
y_test = test["y"]

### 5. Modelos Avaliados e Justificativa de Escolha

Foram avaliados 7 algoritmos de classificação para identificar qual melhor captura os padrões da série:

| Modelo | Característica principal |
|---|---|
| Regressão Logística | Baseline linear simples |
| Decision Tree | Captura não-linearidades, sem ensemble |
| Random Forest | Ensemble de árvores, robusto a overfitting |
| SVM | Margem máxima de separação |
| KNN | Baseado em similaridade local |
| Gradient Boosting | Ensemble sequencial com correção de erros |
| XGBoost | Gradient boosting otimizado, melhor performance |

A métrica principal de avaliação é a **acurácia**, conforme exigido pelo briefing. A matriz de confusão e o relatório de classificação (precisão, recall, F1) são usados como métricas secundárias para entender a qualidade das previsões por classe.

In [911]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

model_log = LogisticRegression(max_iter=1000)
model_log.fit(X_train, y_train)

pred_log = model_log.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_log))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_log))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_log))

Accuracy: 0.5333333333333333

Matriz de confusão:  [[ 0 14]
 [ 0 16]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        14
         1.0       0.53      1.00      0.70        16

    accuracy                           0.53        30
   macro avg       0.27      0.50      0.35        30
weighted avg       0.28      0.53      0.37        30



/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0

In [912]:
from sklearn.tree import DecisionTreeClassifier

model_tree = DecisionTreeClassifier(random_state=42)
model_tree.fit(X_train, y_train)

pred_tree = model_tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_tree))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_tree))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_tree))

Accuracy: 0.36666666666666664

Matriz de confusão:  [[ 8  6]
 [13  3]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.38      0.57      0.46        14
         1.0       0.33      0.19      0.24        16

    accuracy                           0.37        30
   macro avg       0.36      0.38      0.35        30
weighted avg       0.36      0.37      0.34        30



In [913]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_rf))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_rf))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_rf))

Accuracy: 0.36666666666666664

Matriz de confusão:  [[ 2 12]
 [ 7  9]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.22      0.14      0.17        14
         1.0       0.43      0.56      0.49        16

    accuracy                           0.37        30
   macro avg       0.33      0.35      0.33        30
weighted avg       0.33      0.37      0.34        30



In [914]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [915]:
from sklearn.svm import SVC

model_svm = SVC()
model_svm.fit(X_train_scaled, y_train)

pred_svm = model_svm.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_svm))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_svm))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_svm))

Accuracy: 0.3333333333333333

Matriz de confusão:  [[ 4 10]
 [10  6]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.29      0.29      0.29        14
         1.0       0.38      0.38      0.38        16

    accuracy                           0.33        30
   macro avg       0.33      0.33      0.33        30
weighted avg       0.33      0.33      0.33        30



In [916]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)

pred_knn = model_knn.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_knn))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_knn))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_knn))

Accuracy: 0.36666666666666664

Matriz de confusão:  [[ 5  9]
 [10  6]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.33      0.36      0.34        14
         1.0       0.40      0.38      0.39        16

    accuracy                           0.37        30
   macro avg       0.37      0.37      0.37        30
weighted avg       0.37      0.37      0.37        30



In [917]:
from sklearn.ensemble import GradientBoostingClassifier

model_gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
model_gb.fit(X_train, y_train)

pred_gb = model_gb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_gb))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_gb))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_gb))

Accuracy: 0.5333333333333333

Matriz de confusão:  [[8 6]
 [8 8]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.50      0.57      0.53        14
         1.0       0.57      0.50      0.53        16

    accuracy                           0.53        30
   macro avg       0.54      0.54      0.53        30
weighted avg       0.54      0.53      0.53        30



In [918]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42
)
model_xgb.fit(X_train, y_train)

pred_xgb = model_xgb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_xgb))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_xgb))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_xgb))

Accuracy: 0.43333333333333335

Matriz de confusão:  [[5 9]
 [8 8]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.38      0.36      0.37        14
         1.0       0.47      0.50      0.48        16

    accuracy                           0.43        30
   macro avg       0.43      0.43      0.43        30
weighted avg       0.43      0.43      0.43        30



In [919]:
resultados = {
    "Logistic Regression": accuracy_score(y_test, pred_log),
    "Decision Tree": accuracy_score(y_test, pred_tree),
    "Random Forest": accuracy_score(y_test, pred_rf),
    "KNN": accuracy_score(y_test, pred_knn),
    "SVM": accuracy_score(y_test, pred_svm),
    "Gradient Boosting": accuracy_score(y_test, pred_gb),
    "XGBoost": accuracy_score(y_test, pred_xgb)
}

pd.Series(resultados).sort_values(ascending=False)

Logistic Regression    0.533333
Gradient Boosting      0.533333
XGBoost                0.433333
Decision Tree          0.366667
Random Forest          0.366667
KNN                    0.366667
SVM                    0.333333
dtype: float64

In [920]:
print("Distribuição do target no TREINO:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribuição do target no TESTE:")
print(y_test.value_counts(normalize=True).round(3))

print("\nDatas do conjunto de teste:")
print(test.index[0], "até", test.index[-1])
print("Total de dias:", len(test))

Distribuição do target no TREINO:
y
1.0    0.541
0.0    0.459
Name: proportion, dtype: float64

Distribuição do target no TESTE:
y
1.0    0.533
0.0    0.467
Name: proportion, dtype: float64

Datas do conjunto de teste:
2026-01-22 00:00:00 até 2026-03-19 00:00:00
Total de dias: 30


In [921]:
correlacoes = X_train.copy()
correlacoes["y"] = y_train

print("Correlação de cada feature com o target (treino):")
print(correlacoes.corr()["y"].drop("y").sort_values(key=abs, ascending=False).round(4))

Correlação de cada feature com o target (treino):
dist_ma_5            -0.0288
ma_diff_rel           0.0226
retorno_3             0.0205
retorno_1             0.0161
dist_ma_10           -0.0107
pressao_fechamento    0.0080
momentum_5            0.0035
range_diff            0.0008
Name: y, dtype: float64


In [922]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, None],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)

print("Melhores parâmetros:", grid_rf.best_params_)
print("Melhor acurácia no CV:", round(grid_rf.best_score_, 4))

pred_rf_tuned = grid_rf.best_estimator_.predict(X_test)

print("\nAcurácia no teste:", accuracy_score(y_test, pred_rf_tuned))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_rf_tuned))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_rf_tuned))

Melhores parâmetros: {'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 300}
Melhor acurácia no CV: 0.5312

Acurácia no teste: 0.5

Matriz de confusão:  [[ 0 14]
 [ 1 15]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        14
         1.0       0.52      0.94      0.67        16

    accuracy                           0.50        30
   macro avg       0.26      0.47      0.33        30
weighted avg       0.28      0.50      0.36        30



In [923]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

param_grid_xgb = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0]
}

grid_xgb = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
    param_grid=param_grid_xgb,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1
)
grid_xgb.fit(X_train, y_train)

print("Melhores parâmetros:", grid_xgb.best_params_)
print("Melhor acurácia no CV:", round(grid_xgb.best_score_, 4))

pred_xgb_tuned = grid_xgb.best_estimator_.predict(X_test)

print("\nAcurácia no teste:", accuracy_score(y_test, pred_xgb_tuned))
print('\nMatriz de confusão:\n', confusion_matrix(y_test, pred_xgb_tuned))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_xgb_tuned))

/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [20:05:04] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [20:05:04] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/isabellarodriguessandes/projects/previsao-ibovespa-machine-learning/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [20:05:04] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/isabellarodriguessandes/projects/pre

Melhores parâmetros: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}
Melhor acurácia no CV: 0.5319

Acurácia no teste: 0.43333333333333335

Matriz de confusão:
 [[ 7  7]
 [10  6]]

Relatório de Classificação:
               precision    recall  f1-score   support

         0.0       0.41      0.50      0.45        14
         1.0       0.46      0.38      0.41        16

    accuracy                           0.43        30
   macro avg       0.44      0.44      0.43        30
weighted avg       0.44      0.43      0.43        30



### 6. Resultados dos Modelos

Foram avaliados diferentes algoritmos de classificação supervisionada com o objetivo de prever a direção do IBOVESPA no dia seguinte.

O conjunto de teste foi composto pelos **últimos 30 pregões**, respeitando a natureza temporal da série e evitando vazamento de informação.

Os resultados obtidos foram:

- **SVM:** 66,7% de acurácia (20 acertos em 30 observações)
- **XGBoost:** 66,7% de acurácia (20 acertos em 30 observações)
- **Random Forest:** 63,3% de acurácia
- **Logistic Regression:** 60,0% de acurácia

Após ajuste de hiperparâmetros, não houve melhora significativa de desempenho, indicando que o ganho marginal com tuning foi limitado neste conjunto de dados.

De forma geral, os modelos apresentaram capacidade moderada de previsão direcional diária, porém **nenhum atingiu a meta de 75% estabelecida no briefing.**

### 7. Interpretação dos Resultados

A previsão da direção diária de índices acionários é reconhecidamente um problema complexo, pois os preços incorporam rapidamente novas informações e expectativas do mercado.

Mesmo utilizando indicadores técnicos clássicos — como retornos defasados, médias móveis, momentum e amplitude — a previsibilidade direcional diária permaneceu limitada.

Os resultados sugerem que:

- O comportamento do índice apresenta alto nível de ruído no horizonte de 1 dia
- Sinais técnicos isolados possuem poder preditivo moderado
- A combinação de diferentes modelos não foi suficiente para atingir a acurácia desejada

Ainda assim, o desempenho acima de 60% indica que os modelos conseguem capturar parcialmente padrões de curto prazo.

### 8. Limitações

Algumas limitações importantes devem ser consideradas:

- Utilização apenas de **variáveis técnicas baseadas no próprio preço**
- Horizonte de previsão muito curto (1 pregão)
- Pequeno tamanho do conjunto de teste (30 observações)
- Possível influência de fatores macroeconômicos e notícias não capturados pelo modelo

Além disso, o mercado financeiro apresenta características de não estacionariedade, o que dificulta a generalização de padrões ao longo do tempo.

In [924]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

print("Baseline (classe majoritária):", accuracy_score(y_test, pred_dummy))

Baseline (classe majoritária): 0.5333333333333333


In [925]:
from sklearn.metrics import balanced_accuracy_score

print("Balanced Accuracy:", balanced_accuracy_score(y_test, pred_xgb))

Balanced Accuracy: 0.4285714285714286


In [926]:
comparacao = pd.DataFrame({
    "Modelo": resultados.keys(),
    "Acurácia": resultados.values()
}).sort_values("Acurácia", ascending=False)

comparacao

,Modelo,Acurácia
0,Logistic Regression,0.533333
5,Gradient Boosting,0.533333
6,XGBoost,0.433333
1,Decision Tree,0.366667
2,Random Forest,0.366667
3,KNN,0.366667
4,SVM,0.333333


In [927]:
# =========================================================
# 🔵 Download robusto de variáveis externas
# =========================================================

import yfinance as yf
import pandas as pd

inicio = "2015-01-01"

def baixar_serie(ticker, nome):
    df_ext = yf.download(ticker, start=inicio)

    # Se vier MultiIndex
    if isinstance(df_ext.columns, pd.MultiIndex):
        df_ext = df_ext.droplevel(1, axis=1)

    # Detectar coluna correta
    if "Adj Close" in df_ext.columns:
        col = "Adj Close"
    elif "Close" in df_ext.columns:
        col = "Close"
    else:
        raise ValueError(f"Nenhuma coluna de preço encontrada para {ticker}")

    df_ext = df_ext[[col]].rename(columns={col: nome})
    return df_ext

sp500_df = baixar_serie("^GSPC", "sp500")
dolar_df = baixar_serie("BRL=X", "dolar")
vix_df = baixar_serie("^VIX", "vix")
petroleo_df = baixar_serie("CL=F", "petroleo")

print("Dados externos baixados corretamente.")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Dados externos baixados corretamente.


In [928]:
df = df.join(sp500_df, how="left")
df = df.join(dolar_df, how="left")
df = df.join(vix_df, how="left")
df = df.join(petroleo_df, how="left")

df = df.sort_index()
df = df.ffill()

df["ret_sp500"] = df["sp500"].pct_change().shift(1)
df["ret_dolar"] = df["dolar"].pct_change().shift(1)
df["ret_vix"] = df["vix"].pct_change().shift(1)
df["ret_petroleo"] = df["petroleo"].pct_change().shift(1)

df = df.dropna()

In [929]:
# =========================================================
# 🔵 Comparação: modelo original vs modelo com variáveis externas
# =========================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import pandas as pd

# 1. Features originais
features_base = [
    "retorno_1",
    "retorno_3",
    "momentum_5",
    "range_diff",
    "dist_ma_5",
    "dist_ma_10",
    "ma_diff_rel",
    "pressao_fechamento"
]

# 2. Features com variáveis externas
features_macro = features_base + [
    "ret_sp500",
    "ret_dolar",
    "ret_vix",
    "ret_petroleo"
]

# 3. Criar uma cópia apenas com colunas necessárias
df_teste = df.copy()

# Remover linhas com NaN apenas nas colunas usadas
df_teste = df_teste.dropna(subset=features_macro + ["y"])

# 4. Split temporal
train_macro = df_teste.iloc[:-30]
test_macro = df_teste.iloc[-30:]

# Base original
X_train_base = train_macro[features_base]
X_test_base = test_macro[features_base]

# Base com macro
X_train_macro = train_macro[features_macro]
X_test_macro = test_macro[features_macro]

y_train_macro = train_macro["y"]
y_test_macro = test_macro["y"]

# 5. Modelos
modelos = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        random_state=42
    ),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=1, gamma="scale", random_state=42))
    ]),
    
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42
    )
}

# 6. Avaliação
resultados = []

for nome, modelo in modelos.items():
    # cenário base
    modelo.fit(X_train_base, y_train_macro)
    pred_base = modelo.predict(X_test_base)
    acc_base = accuracy_score(y_test_macro, pred_base)
    
    # cenário com macro
    modelo.fit(X_train_macro, y_train_macro)
    pred_macro = modelo.predict(X_test_macro)
    acc_macro = accuracy_score(y_test_macro, pred_macro)
    
    resultados.append({
        "Modelo": nome,
        "Acurácia Base": round(acc_base, 4),
        "Acurácia + Macro": round(acc_macro, 4),
        "Ganho": round(acc_macro - acc_base, 4)
    })

print("Comparação entre cenário original e cenário com variáveis externas:")
display(comparacao)

Comparação entre cenário original e cenário com variáveis externas:


,Modelo,Acurácia
0,Logistic Regression,0.533333
5,Gradient Boosting,0.533333
6,XGBoost,0.433333
1,Decision Tree,0.366667
2,Random Forest,0.366667
3,KNN,0.366667
4,SVM,0.333333


In [930]:
# =========================================================
# ⭐ EXPERIMENTO FINAL OTIMIZADO (isolado do pipeline principal)
# =========================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score


# ---------------------------------------------------------
# 🔵 1. Criar cópia dos dados (não altera o original)
# ---------------------------------------------------------

df_exp = df.copy()


# ---------------------------------------------------------
# 🔵 2. Novo TARGET com zona neutra (reduz ruído direcional)
# ---------------------------------------------------------

ret_futuro = df_exp["Último"].shift(-1) / df_exp["Último"] - 1

df_exp["y_exp"] = np.where(ret_futuro > 0.002, 1,
                  np.where(ret_futuro < -0.002, 0, np.nan))


# ---------------------------------------------------------
# 🔵 3. Novas FEATURES fortes
# ---------------------------------------------------------

# Tendência longa
df_exp["ma_20"] = df_exp["Último"].rolling(20).mean().shift(1)
df_exp["dist_ma_20"] = df_exp["Último"] / df_exp["ma_20"] - 1
df_exp["ma_diff_5_20"] = df_exp["ma_5"] / df_exp["ma_20"] - 1

# RSI
delta = df_exp["Último"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / (avg_loss + 1e-9)

df_exp["rsi_14"] = 100 - (100 / (1 + rs))
df_exp["rsi_14"] = df_exp["rsi_14"].shift(1)


# ---------------------------------------------------------
# 🔵 4. Lista de FEATURES otimizada
# ---------------------------------------------------------

features_exp = [
    "retorno_1",
    "retorno_3",
    "momentum_5",
    "range_diff",
    "dist_ma_5",
    "dist_ma_10",
    "dist_ma_20",
    "ma_diff_rel",
    "ma_diff_5_20",
    "pressao_fechamento",
    "rsi_14",
    "ret_sp500",
    "ret_dolar",
    "ret_vix",
    "ret_petroleo"
]


# ---------------------------------------------------------
# 🔵 5. Limpeza apenas das colunas usadas
# ---------------------------------------------------------

df_exp = df_exp.dropna(subset=features_exp + ["y_exp"])


# ---------------------------------------------------------
# 🔵 6. Split temporal
# ---------------------------------------------------------

train_exp = df_exp.iloc[:-30]
test_exp = df_exp.iloc[-30:]

X_train_exp = train_exp[features_exp]
y_train_exp = train_exp["y_exp"]

X_test_exp = test_exp[features_exp]
y_test_exp = test_exp["y_exp"]


# ---------------------------------------------------------
# 🔵 7. Modelos
# ---------------------------------------------------------

pipe_log = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(C=1.5, max_iter=2000))
])

model_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42
)

ensemble = VotingClassifier(
    estimators=[
        ("log", pipe_log),
        ("xgb", model_xgb)
    ],
    voting="soft"
)


# ---------------------------------------------------------
# 🔵 8. Treinar e avaliar
# ---------------------------------------------------------

modelos_exp = {
    "Logistic Tunado": pipe_log,
    "XGBoost Tunado": model_xgb,
    "Ensemble": ensemble
}

resultados_exp = []

for nome, modelo in modelos_exp.items():
    
    modelo.fit(X_train_exp, y_train_exp)
    pred = modelo.predict(X_test_exp)
    
    acc = accuracy_score(y_test_exp, pred)
    
    resultados_exp.append({
        "Modelo": nome,
        "Acurácia": round(acc, 4)
    })


resultado_final_exp = pd.DataFrame(resultados_exp).sort_values(
    "Acurácia", ascending=False
)

print("\n⭐ RESULTADO EXPERIMENTO OTIMIZADO")
display(resultado_final_exp)


⭐ RESULTADO EXPERIMENTO OTIMIZADO


,Modelo,Acurácia
0,Logistic Tunado,0.5333
2,Ensemble,0.5000
1,XGBoost Tunado,0.3667


Ao ampliar o período histórico para aproximadamente 10 anos, observou-se redução significativa da acurácia dos modelos.
Esse comportamento sugere que padrões técnicos de curto prazo não se mantêm estáveis ao longo do tempo, refletindo a natureza dinâmica e adaptativa do mercado financeiro.